In [21]:
import pandas as pd
import re

In [22]:
df = pd.read_json('result.json')

if 'assignments' in df.columns:
    df = pd.json_normalize(df['assignments'])

if 'score' in df.columns:
    df = df[['date', 'teamMemberId', 'shiftTypeId', 'score']]
else:
    df = df[['date', 'teamMemberId', 'shiftTypeId']]

In [23]:
members = df['teamMemberId'].unique()
print(members)

<StringArray>
['tmptf003-0000-0000-0000-000000000008',
 'tmptf002-0000-0000-0000-000000000007',
 'tmptf001-0000-0000-0000-000000000006',
 'tmstf004-0000-0000-0000-000000000005',
 'tmstf003-0000-0000-0000-000000000004',
 'tmstf002-0000-0000-0000-000000000003',
 'tmstf001-0000-0000-0000-000000000002',
 'tmadm000-0000-0000-0000-000000000001']
Length: 8, dtype: str


In [24]:
prefix_map = {
    "ptf": "P",
    "stf": "F",
    "adm": "M"
}

member_map = {}
for member in members:
    # remove 'tm' prefix
    id_core = member[2:8]
    # get sections
    letters = id_core[:3]
    number = id_core[3:]
    # remove leading zeros
    number = str(int(number))
    member_map[member] = prefix_map[letters] + number

print(member_map)

{'tmptf003-0000-0000-0000-000000000008': 'P3', 'tmptf002-0000-0000-0000-000000000007': 'P2', 'tmptf001-0000-0000-0000-000000000006': 'P1', 'tmstf004-0000-0000-0000-000000000005': 'F4', 'tmstf003-0000-0000-0000-000000000004': 'F3', 'tmstf002-0000-0000-0000-000000000003': 'F2', 'tmstf001-0000-0000-0000-000000000002': 'F1', 'tmadm000-0000-0000-0000-000000000001': 'M0'}


In [25]:
shifts = df['shiftTypeId'].unique()

shift_map = {}
for shift in shifts:
    # Take first 7 chars (e.g., 'stP00000')
    code = shift[:7]
    # Extract only capital letters
    letters = ''.join(re.findall(r'[A-Z]+', code))
    shift_map[shift] = letters

print(shift_map)

{'stA00000-0000-0000-0000-000000000002': 'A', 'stBB0000-0000-0000-0000-000000000004': 'BB', 'stAA0000-0000-0000-0000-000000000001': 'AA', 'stB00000-0000-0000-0000-000000000003': 'B', 'stP00000-0000-0000-0000-000000000005': 'P'}


In [26]:
df['teamMemberId'] = df['teamMemberId'].map(member_map)
df['shiftTypeId'] = df['shiftTypeId'].map(shift_map)

In [27]:
df['date'] = pd.to_datetime(df['date'])
df['date'] = df['date'].dt.tz_convert('Asia/Tokyo')
df['date'] = df['date'].dt.date
display(df)

,date,teamMemberId,shiftTypeId
0,2026-03-01,P3,A
1,2026-03-02,P3,A
2,2026-03-08,P3,BB
3,2026-03-09,P3,BB
4,2026-03-15,P3,AA
...,...,...,...
137,2026-03-24,M0,A
138,2026-03-25,M0,A
139,2026-03-27,M0,BB
140,2026-03-28,M0,BB


In [28]:
dfs = df.sort_values(
    by=['date', 'teamMemberId', 'shiftTypeId']
).reset_index(drop=True)

display(dfs)

,date,teamMemberId,shiftTypeId
0,2026-03-01,F2,AA
1,2026-03-01,F4,A
2,2026-03-01,M0,BB
3,2026-03-01,P1,BB
4,2026-03-01,P2,BB
...,...,...,...
137,2026-03-30,P3,B
138,2026-03-31,F1,A
139,2026-03-31,F2,A
140,2026-03-31,P1,BB


In [29]:
grouped = df.groupby('date')
for date, group in grouped:
    display(group.sort_values(['teamMemberId', 'shiftTypeId']))

,date,teamMemberId,shiftTypeId
76,2026-03-01,F2,AA
32,2026-03-01,F4,A
120,2026-03-01,M0,BB
20,2026-03-01,P1,BB
10,2026-03-01,P2,BB
0,2026-03-01,P3,A


,date,teamMemberId,shiftTypeId
54,2026-03-02,F3,AA
121,2026-03-02,M0,BB
11,2026-03-02,P2,BB
1,2026-03-02,P3,A


,date,teamMemberId,shiftTypeId
77,2026-03-03,F2,A
55,2026-03-03,F3,AA
33,2026-03-03,F4,BB
122,2026-03-03,M0,BB


,date,teamMemberId,shiftTypeId
98,2026-03-04,F1,BB
56,2026-03-04,F3,AA
34,2026-03-04,F4,BB
21,2026-03-04,P1,P


,date,teamMemberId,shiftTypeId
99,2026-03-05,F1,BB
57,2026-03-05,F3,AA
35,2026-03-05,F4,BB
123,2026-03-05,M0,AA


...

,date,teamMemberId,shiftTypeId
116,2026-03-27,F1,A
94,2026-03-27,F2,BB
51,2026-03-27,F4,BB
139,2026-03-27,M0,BB


,date,teamMemberId,shiftTypeId
117,2026-03-28,F1,A
95,2026-03-28,F2,BB
73,2026-03-28,F3,AA
52,2026-03-28,F4,BB
140,2026-03-28,M0,BB
29,2026-03-28,P1,P


,date,teamMemberId,shiftTypeId
118,2026-03-29,F1,A
96,2026-03-29,F2,BB
74,2026-03-29,F3,AA
30,2026-03-29,P1,P
18,2026-03-29,P2,BB
8,2026-03-29,P3,B


,date,teamMemberId,shiftTypeId
75,2026-03-30,F3,AA
53,2026-03-30,F4,A
141,2026-03-30,M0,BB
9,2026-03-30,P3,B


,date,teamMemberId,shiftTypeId
119,2026-03-31,F1,A
97,2026-03-31,F2,A
31,2026-03-31,P1,BB
19,2026-03-31,P2,P


In [30]:
count_table = df.groupby('date').size().reset_index(name='count')
display(count_table)

,date,count
0,2026-03-01,6
1,2026-03-02,4
2,2026-03-03,4
3,2026-03-04,4
4,2026-03-05,4
5,2026-03-06,4
6,2026-03-07,6
7,2026-03-08,6
8,2026-03-09,4
9,2026-03-10,4
